In [ ]:
import pandas as pd
import numpy as np

class ZoneModel:
    def __init__(self, zone_name, **kwargs):
        self.zone_name = zone_name
        
        # --- Unpack all inputs from the orchestrator ---
        # Dataframe storage
        self.hourly_df = kwargs.get('hourly_df') # Storing the DF here
        
        # Schedules
        self.occ_profile = kwargs.get('occ_profile')
        self.equip_profile = kwargs.get('equip_profile')
        self.vent_profile = kwargs.get('vent_profile')
        self.system_profile = kwargs.get('system_profile')

        # Geometry
        self.floor_area = kwargs.get('floor_area')
        self.room_height = kwargs.get('room_height')
        self.room_volume = self.floor_area * self.room_height
        self.area_roof = kwargs.get('area_roof')
        self.area_ground = kwargs.get('area_ground')
        self.total_facade_areas = kwargs.get('total_facade_areas')
        self.window_percentages = kwargs.get('window_percentages')
        self.glazing_percentages = kwargs.get('glazing_percentages')

        # Thermal Properties
        self.rc_facade = kwargs.get('rc_facade')
        self.rc_roof = kwargs.get('rc_roof')
        self.rc_ground_floor = kwargs.get('rc_ground_floor')
        self.u_value_windows = kwargs.get('u_value_windows')

        # Solar & Ventilation Constants
        self.alfai = kwargs.get('alfai')
        self.alfao = kwargs.get('alfao')
        
        self.solar_absorption_coefficient = kwargs.get('solar_absorption_coefficient')
        self.solar_heat_coefficient_shading = kwargs.get('solar_heat_coefficient_shading')
        self.solar_heat_coefficient_glazing = kwargs.get('solar_heat_coefficient_glazing')

        self.air_density = kwargs.get('air_density')
        self.air_heat_capacity = kwargs.get('air_heat_capacity')
        self.vent_flow_per_person = kwargs.get('vent_flow_per_person')
        self.infiltration_ach = kwargs.get('infiltration_ach')
        self.natural_vent_rate = kwargs.get('natural_vent_rate')
        self.vent_cooling_setpoint = kwargs.get('vent_cooling_setpoint')

        # Internal Gains & System Constants
        self.max_people_per_m2 = kwargs.get('max_people_per_m2')
        self.heat_per_person = kwargs.get('heat_per_person')
        self.appliances_w_m2 = kwargs.get('appliances_w_m2')
        self.lighting_w_m2 = kwargs.get('lighting_w_m2')

        # --- HVAC & Ventilation System ---
        self.system_pressure_drop = kwargs.get('system_pressure_drop')
        self.efficiency_fan_and_motor = kwargs.get('efficiency_fan_and_motor')
        self.eta = kwargs.get('eta', 0.0) 

        # Simulation Constants
        self.heating_setpoint = kwargs.get('heating_setpoint')
        self.cooling_setpoint = kwargs.get('cooling_setpoint')

        self.thermal_mass_factor = kwargs.get('thermal_mass_factor', 165000) # Default to heavy if not provided
        self.thermal_capacity = self.thermal_mass_factor * self.floor_area
        self.dt = 3600
        self.t_ground = kwargs.get('t_ground')

        self.heating_power_max = kwargs.get('heating_power_max')
        self.cooling_power_max = kwargs.get('cooling_power_max')

        # Simulation starting temperature
        self.current_temp = kwargs.get('initial_temp', 20.0) # Need a starting temp

        # Pre-calculate surface areas for the loop
        self.solar_orientations = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
        self._initialize_surfaces()
        
    def _initialize_surfaces(self):
        self.area_windows = 0
        self.area_walls = 0
        self.glass_areas = {}
        self.opaque_wall_areas = {}

        for orient in self.solar_orientations:
            facade = self.total_facade_areas.get(orient, 0.0)
            win_perc = self.window_percentages.get(orient, 0.0)
            glaz_perc = self.glazing_percentages.get(orient, 1.0)
    
            window_area = facade * win_perc
            self.glass_areas[orient] = window_area * glaz_perc
            self.opaque_wall_areas[orient] = facade - window_area
            
            self.area_windows += window_area
            self.area_walls += self.opaque_wall_areas[orient]
            
        # Extra required calculations
        self.max_people = self.floor_area * self.max_people_per_m2
        self.flow_rate_vent = self.max_people * self.vent_flow_per_person # The same as total_flow_rate_vent
        self.total_flow_rate_vent = self.vent_flow_per_person * self.max_people_per_m2 * self.floor_area # The same as flow_rate_vent
        self.fans_power = (self.total_flow_rate_vent * self.system_pressure_drop) / (3600 * self.efficiency_fan_and_motor)
        
        # Tracking lists for results (outside loop to avoid reinitialization, since this simulation is per timestep)
        self.annual_heating_results = []
        self.annual_cooling_results = []
        self.annual_temp_results = []

    def calculate_nta8800_u_value(self, rc_value):
        if rc_value <= 0: return 0
        delta_u = 0.15 # Either use delta_u = max(0, 0.1 - 0.25 * (u_undisturbed - 0.4)) for accuracy, or the 0.15 fixed penalty as is done in the excel.
        return (1 / (rc_value + 0.17)) + delta_u

    def calculate_hour_step(self, t, external_q_flow=0):
        """Processes one single hour of physics."""
        # --- PRE-LOOP CALCULATIONS (now inside step) ---
        u_value_walls = self.calculate_nta8800_u_value(self.rc_facade)        
        u_value_roof  = self.calculate_nta8800_u_value(self.rc_roof) 
        u_value_ground = self.calculate_nta8800_u_value(self.rc_ground_floor)       

        trans_e_windows = self.area_windows * self.u_value_windows
        trans_e_walls = self.area_walls * u_value_walls
        trans_e_roof = self.area_roof * u_value_roof
        H_e_transmission = trans_e_walls + trans_e_roof + trans_e_windows

        # --- DATA FETCHING ---
        row = self.hourly_df.iloc[t]
        idx_168 = t % 168
        current_occ_val = self.occ_profile[idx_168]
        current_light_val = self.equip_profile[idx_168]
        current_vent_val = self.vent_profile[idx_168]
        current_sys = self.system_profile[idx_168]
        current_temp = self.current_temp

        # --- SOLAR GAINS ---
        glazing_gains_breakdown = {}
        opaque_gains_breakdown = {}

        for orient in self.solar_orientations:
            if current_temp >= self.vent_cooling_setpoint and row[orient] > 0:
                active_shading = self.solar_heat_coefficient_shading
            else:
                active_shading = 1.0

            glazing_val = (row[orient] * self.glass_areas[orient] * self.solar_heat_coefficient_glazing * active_shading)
            glazing_gains_breakdown[orient] = glazing_val

            opaque_val = (row[orient] * self.opaque_wall_areas[orient] * self.solar_absorption_coefficient * (u_value_walls / self.alfao))
            opaque_gains_breakdown[orient] = opaque_val

        sun_glazing = sum(glazing_gains_breakdown.values())
        sun_opaque_walls = sum(opaque_gains_breakdown.values())
        sun_opaque_roof = row['Horizontal'] * self.solar_absorption_coefficient * (u_value_roof / self.alfao) * self.area_roof

        # --- INTERNAL GAINS ---
        people_heat = self.max_people * current_occ_val * self.heat_per_person
        lighting_heat = self.lighting_w_m2 * self.floor_area * current_light_val
        equipment_heat = self.appliances_w_m2 * self.floor_area * current_light_val
        fans_heat = 0.5 * self.fans_power * current_vent_val                                        # Assuming 50% of fan power converts to heat in the space 
       
        total_internal_gains = people_heat + equipment_heat + lighting_heat + (fans_heat * 0.5)     # The excel adds another 0.5 factor (not sure why?)

        # --- TOTAL FREE GAINS (Including External Flow from Orchestrator) ---
        total_free_gains = sun_glazing + sun_opaque_walls + sun_opaque_roof + total_internal_gains + external_q_flow # NEW: Heat from one zone to another (shared interal walls)

        # --- GROUND AND INFILTRATION ---
        flow_rate_inf = self.room_volume * self.infiltration_ach

        h_inf = (flow_rate_inf * self.air_density * self.air_heat_capacity) / 3600
        h_ground = self.area_ground * u_value_ground

        # --- VENTILATION ---
        current_flow_m3h = self.flow_rate_vent * current_occ_val * current_vent_val 

        h_vent_heat = (1 - self.eta) * (self.flow_rate_vent * current_vent_val * self.air_density * self.air_heat_capacity) / 3600
        h_vent_cool = ((self.flow_rate_vent * current_vent_val) + (self.natural_vent_rate * self.room_volume)) * self.air_density * self.air_heat_capacity / 3600

        # --- TOTAL HEAT TRANSFER ---
        t_ext = row['T'] 

        H_total_heat = H_e_transmission + h_vent_heat + h_inf        
        H_total_cool = H_e_transmission + h_vent_cool + h_inf  

        Total_exterior_heat = (H_total_heat * t_ext) + (h_ground * self.t_ground)
        Total_exterior_cool = (H_total_cool * t_ext) + (h_ground * self.t_ground)

        H_total_with_ground_heat = H_total_heat + h_ground
        H_total_with_ground_cool = H_total_cool + h_ground
        
        # --- DYNAMIC VENTILATION LOGIC ---
        den_test = 1 + (self.dt / self.thermal_capacity) * H_total_with_ground_heat

        t_check = (current_temp + (self.dt / self.thermal_capacity) * (total_free_gains + Total_exterior_heat)) / den_test

        t_eq_heat = (total_free_gains + Total_exterior_heat) / H_total_with_ground_heat
        t_eq_cool = (total_free_gains + Total_exterior_cool) / H_total_with_ground_cool

        if t_check > self.vent_cooling_setpoint and t_ext < current_temp:
            H_active = H_total_with_ground_cool
            self.current_ach = (current_flow_m3h / self.room_volume) + self.natural_vent_rate 
        else:
            H_active = H_total_with_ground_heat
            self.current_ach = (current_flow_m3h / self.room_volume) 

        # --- EXPONENTIALS ---
        exp_factor_heat = 1 - np.exp(-(H_total_with_ground_heat) / self.thermal_capacity * self.dt)        
        exp_factor_cool = 1 - np.exp(-(H_total_with_ground_cool) / self.thermal_capacity * self.dt)        
        # Or combine it into the following: exp_factor_active = 1 - np.exp(-(H_active) / self.thermal_capacity * self.dt)

        # --- SCOUTING TEMPERATURES ---
        t_room_free_heating = current_temp + (t_eq_heat - current_temp) * exp_factor_heat
        t_room_free_cooling = current_temp + (t_eq_cool - current_temp) * exp_factor_cool

        t_room_max_heating = t_room_free_heating + (self.heating_power_max / H_total_with_ground_heat) * exp_factor_heat
        t_room_max_cooling = t_room_free_cooling + (self.cooling_power_max / H_total_with_ground_cool) * exp_factor_cool

        # --- CONTROL LOGIC GATE ---
        q_heat_kwh = 0.0
        q_cool_kwh = 0.0

        system_active = current_sys

        if system_active > 0:
            # STEP 1: Heating Capacity Limit (Too cold even with max heat)
            if t_room_max_heating < self.heating_setpoint:
                current_temp = t_room_max_heating
                q_heat_kwh = self.heating_power_max / 1000.0

            # STEP 2: Heating Setpoint Reached (Modulating power)
            elif t_room_max_heating >= self.heating_setpoint and t_room_free_heating < self.heating_setpoint:
                current_temp = self.heating_setpoint
                p_needed = H_total_with_ground_heat * (self.heating_setpoint - t_room_free_heating) / exp_factor_heat
                q_heat_kwh = p_needed / 1000.0

            # STEP 3: Free Floating - Standard Ventilation (HVAC is idle)
            elif t_room_free_heating >= self.heating_setpoint and t_room_free_heating <= self.vent_cooling_setpoint:
                current_temp = t_room_free_heating

            # STEP 4: Ventilation Setpoint Reached (Windows/Bypass modulating to hold 23.9)
            elif t_room_free_heating > self.vent_cooling_setpoint and t_room_free_cooling < self.vent_cooling_setpoint:
                current_temp = self.vent_cooling_setpoint

            # STEP 5: Free Floating - High Ventilation (Night cooling active but not enough)
            elif t_room_free_cooling >= self.vent_cooling_setpoint and t_room_free_cooling <= self.cooling_setpoint:
                current_temp = t_room_free_cooling

            # STEP 6: Cooling Setpoint Reached (Modulating chiller)
            elif t_room_max_cooling <= self.cooling_setpoint and t_room_free_cooling > self.cooling_setpoint:
                current_temp = self.cooling_setpoint

                if t_ext < self.cooling_setpoint:
                    # Natural vent helps; use the higher coupling
                    p_needed = H_total_with_ground_cool * (self.cooling_setpoint - t_room_free_cooling) / exp_factor_cool
                else:
                    # Natural vent hurts; Excel logic typically assumes windows closed during active chilling
                    p_needed = H_total_with_ground_heat * (self.cooling_setpoint - t_room_free_heating) / exp_factor_heat
                q_cool_kwh = p_needed / 1000.0

            # STEP 7: Cooling Capacity Limit (Too hot even with max cooling)
            elif t_room_max_cooling > self.cooling_setpoint:
                current_temp = t_room_max_cooling
                q_cool_kwh = self.cooling_power_max / 1000.0
        else:
            # System is OFF (e.g., Night/Weekend schedule)
            current_temp = t_room_free_heating

        # Update state and store
        self.current_temp = current_temp
        self.annual_heating_results.append(q_heat_kwh)
        self.annual_cooling_results.append(q_cool_kwh)
        self.annual_temp_results.append(current_temp)
        return current_temp

    def run_simulation(self):
        """Can still be used for standalone tests of one zone."""
        # Reset results
        self.annual_heating_results = []
        self.annual_cooling_results = []
        self.annual_temp_results = []
        
        for t in range(len(self.hourly_df)):
            self.calculate_hour_step(t, external_q_flow=0)

        # --- OUTPUT SECTION --- 
        print(f"\n--- Results for {self.zone_name} ---")
        print(f"Simulation Complete.")
        print(f"Room Volume: {self.room_volume:.2f} m3")
        print(f"Peak Heating: {max(self.annual_heating_results):.2f} kW | Peak Cooling: {min(self.annual_cooling_results):.2f} kW")
        print(f"Total Annual Heat Demand: {sum(self.annual_heating_results):.2f} kWh")
        print(f"Total Annual Cool Demand: {sum(self.annual_cooling_results):.2f} kWh")

        return sum(self.annual_heating_results), sum(self.annual_cooling_results)